In [21]:
import numpy as np
import cv2
from collections import deque
from tkinter import Tk
from tkinter.filedialog import askopenfilename

Tk().withdraw()

source_name = input("For file search, press enter or 0 for webcam: ")

if source_name == '0':
    source_name = 0
else:
    source_name = askopenfilename(
        title="Select video file",
        filetypes=[("MP4 files", "*.mp4"), ("All files", "*.*")]
    )
    if not source_name:
        print("No file selected, exiting.")
        exit()

debug = False

MAG_THRESHOLD = 2.0
MIN_AREA = 1000
TRACK_LENGTH = 10
ERR_RATIO = 0.3

positions = deque(maxlen=TRACK_LENGTH)

def draw_flow(img, flow, step=16):
    h, w = img.shape[:2]
    y, x = np.mgrid[step//2:h:step, step//2:w:step].reshape(2,-1).astype(int)
    fx, fy = flow[y,x].T
    lines = np.vstack([x, y, x-fx, y-fy]).T.reshape(-1, 2, 2)
    lines = np.int32(lines)
    img_bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.polylines(img_bgr, lines, 0, (0,255,0))
    return img_bgr

def draw_hsv(flow):
    fx, fy = flow[:,:,0], flow[:,:,1]
    mag, ang = cv2.cartToPolar(fx, fy)
    hsv = np.zeros((flow.shape[0], flow.shape[1], 3), np.uint8)
    hsv[...,0] = ang * 180 / np.pi / 2
    hsv[...,1] = 255
    hsv[...,2] = np.clip(mag * 4, 0, 255)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

# def is_curved_path(points):
#     pts = np.array(points)
#     x = pts[:, 0]
#     y = pts[:, 1]

#     a, b, c = np.polyfit(x, y, 2)
#     return abs(a) > CURVATURE_THRESHOLD

def velocities(points):
    pts = np.array(points)
    vx = np.diff(pts[:,0])
    vy = np.diff(pts[:,1])
    return vx, vy

def vertical_acceleration(vy):
    ay = np.diff(vy)
    return ay

def is_projectile(points, vx_thresh=2.0, ay_thresh=0.05):
    vx, vy = velocities(points)
    ay = vertical_acceleration(vy)

    if np.std(vx) > vx_thresh:
        return False

    if abs(np.mean(ay)) < ay_thresh:
        return False

    if not (np.any(vy>0) and np.any(vy<0)):
        return False

    return True

def is_curved_path(points, err_ratio=0.2):
    pts = np.array(points)
    x = pts[:, 0]
    y = pts[:, 1]

    p1 = np.polyfit(x, y, 1)
    y1 = p1[0]*x + p1[1]
    err1 = np.mean((y - y1)**2)

    p2 = np.polyfit(x, y, 2)
    y2 = p2[0]*x*x + p2[1]*x + p2[2]
    err2 = np.mean((y - y2)**2)

    return err2 < err_ratio * err1


cap = cv2.VideoCapture(source_name)
if not cap.isOpened():
    print("Error: Could not open video.")
    exit()
    
ret, prev = cap.read()
if not ret:
    print(f"Error: Could not read first frame from {source_name}")
    cap.release()
    exit()

prevgray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)

while True:
    ret, img = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    flow = cv2.calcOpticalFlowFarneback(
        prevgray, gray, None,
        pyr_scale=0.5, 
        levels=3, 
        winsize=15, 
        iterations=3, 
        poly_n=5, 
        poly_sigma=1.2, 
        flags=0
    )
    prevgray = gray

    mag, _ = cv2.cartToPolar(flow[...,0], flow[...,1])

    mask = mag > MAG_THRESHOLD
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((7,7), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((7,7), np.uint8))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        c = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(c)

        if area > MIN_AREA:
            x, y, w, h = cv2.boundingRect(c)
            cx, cy = x + w//2, y + h//2

            if positions:
                prev_cx, prev_cy = positions[-1]
                dist_x, dist_y = cx - prev_cx, cy - prev_cy
                if abs(dist_x) < 150 and abs(dist_y) < 150:
                    positions.append((cx, cy))
            else:
                positions.append((cx, cy))

            cv2.rectangle(img, (x,y), (x+w,y+h), (0,255,0), 2)
            cv2.circle(img, (cx,cy), 5, (0,0,255), -1)
    
    if len(positions) > TRACK_LENGTH // 2:
        if (not debug and is_projectile(positions)) or (debug and is_curved_path(positions, ERR_RATIO)):
            for i in range(1, len(positions)):
                cv2.line(img, positions[i-1], positions[i], (0,0,255), 2)

            cv2.putText(
                img,
                "ELHAJITOTT TEST",
                org=(30,40),
                fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                fontScale=1,
                color=(0,0,255),
                thickness=2
            )

    cv2.imshow("window", img)

    # if debug:
    #     cv2.imshow("Optical Flow HSV", draw_hsv(flow))
    #     cv2.imshow("Optical Flow", draw_flow(gray, flow))

    if cv2.waitKey(5) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [32]:
import cv2
import numpy as np

w, h = 640, 480
fps = 30
frames = 120
FALL_START_FRAME = 40


fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("video1.mp4", fourcc, fps, (w, h))

x = 100
y = 200
vx = 4
vy = 0
g = 0.5

for i in range(frames):
    frame = np.ones((h, w, 3), dtype=np.uint8) * 255

    x += vx

    if i >= FALL_START_FRAME:
        y += vy
        vy += g

    cv2.circle(frame, (int(x), int(y)), 10, (0, 0, 0), -1)
    out.write(frame)

out.release()
